# Практическая работа №6: Подготовка фич и интеграция источников
**Вариант 4**

Задания варианта:
1. Реализовать частотное кодирование `status` и `event_type`.
2. Использовать эти частоты как признаки в агрегатах по пользователю.
3. Сравнить размер и структуру фичей с вариантом one-hot.

In [1]:
import pandas as pd
import numpy as np

## Загрузка данных

In [2]:
orders_files = [
    "data/orders_2024_01.csv",
    "data/orders_2024_02.csv",
    "data/orders_2024_03.csv",
]

df_orders = pd.concat([pd.read_csv(f, parse_dates=["order_datetime"]) for f in orders_files], ignore_index=True)
df_orders["revenue"] = df_orders["price"] * df_orders["quantity"]

print(df_orders.head())

   order_id  user_id      order_datetime  product_id     price  quantity  \
0         1     1044 2024-01-03 18:39:26          26  14038.15         1   
1         2     1079 2024-01-03 17:45:43          16   9172.60         1   
2         3     1041 2024-01-06 22:46:38          25   9035.26         2   
3         4     1083 2024-01-08 21:03:51           9  15234.33         3   
4         5     1068 2024-01-03 23:26:53          24   4134.38         2   

     status   revenue  
0  refunded  14038.15  
1      paid   9172.60  
2      paid  18070.52  
3      paid  45702.99  
4      paid   8268.76  


In [3]:
df_events_records = pd.read_json("data/events_records.json").T.reset_index(drop=True)
df_events_records.columns = ["event_id", "user_id", "event_time", "event_type", "page"]

df_events_lines = pd.read_json("data/events_lines.json", lines=True)

df_events = pd.concat([df_events_records, df_events_lines], ignore_index=True)
df_events = df_events.astype({"event_id": "int64", "user_id": "int64", "event_type": "string", "page": "string"})
df_events["event_time"] = pd.to_datetime(df_events["event_time"])

print(df_events.head())

   event_id  user_id          event_time   event_type         page
0         1     1085 2024-01-30 20:37:32    page_view  /product/26
1         2     1006 2024-02-24 22:55:13  add_to_cart  /product/28
2         3     1024 2024-02-14 17:49:43        click  /product/27
3         4     1062 2024-02-16 11:21:32        click   /product/1
4         5     1012 2024-02-08 05:25:21  add_to_cart            /


## Задание 1. Частотное кодирование `status` и `event_type`

In [4]:
status_freq = df_orders["status"].value_counts(normalize=True)
df_orders["status_freq"] = df_orders["status"].map(status_freq).astype("float64")

event_type_freq = df_events["event_type"].value_counts(normalize=True)
df_events["event_type_freq"] = df_events["event_type"].map(event_type_freq).astype("float64")

print(df_orders[["order_id", "status", "status_freq"]].head())
print()
print(df_events[["event_id", "event_type", "event_type_freq"]].head())

   order_id    status  status_freq
0         1  refunded     0.050400
1         2      paid     0.802133
2         3      paid     0.802133
3         4      paid     0.802133
4         5      paid     0.802133

   event_id   event_type  event_type_freq
0         1    page_view          0.20100
1         2  add_to_cart          0.19550
2         3        click          0.20475
3         4        click          0.20475
4         5  add_to_cart          0.19550


## Задание 2. Частоты как признаки в агрегатах по пользователю

In [5]:
orders_by_user = df_orders.groupby("user_id").agg(
    orders_count=("order_id", "nunique"),
    revenue_sum=("revenue", "sum"),
    status_freq_mean=("status_freq", "mean"),
)

events_by_user = df_events.groupby("user_id").agg(
    events_total=("event_id", "count"),
    event_type_freq_mean=("event_type_freq", "mean"),
)

user_features_freq = orders_by_user.join(events_by_user, how="outer").reset_index()

print(user_features_freq.head())

   user_id  orders_count  revenue_sum  status_freq_mean  events_total  \
0     1001           177   4435767.09          0.667744            54   
1     1002           123   2830004.52          0.680304            41   
2     1003           156   4025266.18          0.703123            39   
3     1004           147   3788030.56          0.700854            37   
4     1005           141   3185653.37          0.679349            44   

   event_type_freq_mean  
0              0.200898  
1              0.200238  
2              0.200218  
3              0.199662  
4              0.199665  


## Задание 3. Сравнение с one-hot кодированием

In [6]:
status_dummies = pd.get_dummies(df_orders["status"], prefix="status", dtype="int64")
event_dummies = pd.get_dummies(df_events["event_type"], prefix="event", dtype="int64")

orders_ohe_by_user = pd.concat([df_orders[["user_id"]], status_dummies], axis=1).groupby("user_id").sum()
events_ohe_by_user = pd.concat([df_events[["user_id"]], event_dummies], axis=1).groupby("user_id").sum()

user_features_ohe = (
    orders_by_user[["orders_count", "revenue_sum"]]
    .join(orders_ohe_by_user, how="outer")
    .join(events_by_user[["events_total"]], how="outer")
    .join(events_ohe_by_user, how="outer")
    .reset_index()
)

print(user_features_ohe.head())

   user_id  orders_count  revenue_sum  status_cancelled  status_paid  \
0     1001           177   4435767.09                26          142   
1     1002           123   2830004.52                16          101   
2     1003           156   4025266.18                19          133   
3     1004           147   3788030.56                17          125   
4     1005           141   3185653.37                23          115   

   status_refunded  events_total  event_add_to_cart  event_click  \
0                9            54                  7           12   
1                6            41                  7            7   
2                4            39                  6            6   
3                5            37                  8            3   
4                3            44                 10            6   

   event_page_view  event_purchase  event_scroll  
0               13              16             6  
1                8              12             7  
2    

In [7]:
comparison = pd.DataFrame({
    "variant": ["frequency", "one-hot"],
    "rows": [user_features_freq.shape[0], user_features_ohe.shape[0]],
    "columns": [user_features_freq.shape[1], user_features_ohe.shape[1]],
    "memory_kb": [
        user_features_freq.memory_usage(deep=True).sum() / 1024,
        user_features_ohe.memory_usage(deep=True).sum() / 1024,
    ],
})

print(comparison.round(2))
print()
print("Frequency:", list(user_features_freq.columns))
print("One-hot:", list(user_features_ohe.columns))

     variant  rows  columns  memory_kb
0  frequency   100        6       4.82
1    one-hot   100       12       9.50

Frequency: ['user_id', 'orders_count', 'revenue_sum', 'status_freq_mean', 'events_total', 'event_type_freq_mean']
One-hot: ['user_id', 'orders_count', 'revenue_sum', 'status_cancelled', 'status_paid', 'status_refunded', 'events_total', 'event_add_to_cart', 'event_click', 'event_page_view', 'event_purchase', 'event_scroll']
